# OpenCloud-SRE: Autonomous GRPO Training

Complete training pipeline using **Unsloth** and **GRPO**.

**Required:** Go to **Runtime -> Change runtime type -> T4 GPU** before running.

Links:
- GitHub: https://github.com/hitendras510/OpenCloud-SRE
- Live Demo: https://huggingface.co/spaces/hitendras510/OpenCloud-SRE

In [ ]:
# Cell 1: Install Dependencies
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q trl transformers accelerate datasets bitsandbytes wandb matplotlib
print('Done installing dependencies.')

In [ ]:
# Cell 2: GPU Verification — MUST pass before loading model
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        'NO GPU DETECTED! '
        'Go to: Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU '
        'Then click Runtime -> Run all'
    )

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print('GPU confirmed:', gpu_name, '| VRAM:', round(vram_gb, 1), 'GB')

In [ ]:
# Cell 3: Load Model with 4-bit QLoRA (Unsloth)
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='Qwen/Qwen2.5-1.5B-Instruct',
    max_seq_length=1024,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print('Model + LoRA adapters loaded successfully!')

In [ ]:
# Cell 4: OpenCloud Simulation Environment
from dataclasses import dataclass

@dataclass
class CloudState:
    traffic: float = 20.0
    temp: float = 30.0
    health: float = 90.0

    def slo_score(self):
        dist = ((self.traffic - 20) ** 2 + (self.temp - 30) ** 2 + (self.health - 90) ** 2) ** 0.5
        return max(0.0, 1.0 - dist / 150.0)

class OpenCloudEnv:
    def reset(self):
        return CloudState(traffic=90.0, temp=85.0, health=10.0)  # CRITICAL state

    def step(self, action):
        if action == 'scale_out':
            return CloudState(20.0, 30.0, 95.0), 1.0   # full recovery
        elif action == 'noop':
            return CloudState(95.0, 90.0, 5.0), -0.5   # penalise doing nothing
        return CloudState(70.0, 60.0, 40.0), 0.3       # partial fix

env = OpenCloudEnv()
state = env.reset()
print('Initial SLO score:', round(state.slo_score(), 3), '(0=degraded, 1=healthy)')

In [ ]:
# Cell 5: GRPO Helpers — Prompt Builder + Advantage Calculator
import json as _json

VALID_ACTIONS = ['throttle_traffic', 'load_balance', 'schema_failover',
                 'cache_flush', 'circuit_breaker', 'restart_pods', 'scale_out', 'noop']

def build_prompt(state):
    return (
        'You are an autonomous SRE AI. Respond ONLY with JSON.\n'
        'Traffic_Load: ' + str(round(state.traffic, 1)) + '\n'
        'DB_Temperature: ' + str(round(state.temp, 1)) + '\n'
        'Network_Health: ' + str(round(state.health, 1)) + '\n'
        'SLO_Score: ' + str(round(state.slo_score(), 3)) + '\n'
        'Valid actions: ' + str(VALID_ACTIONS) + '\n\n'
        'Your JSON ({"intent": "<action>", "confidence": <0-1>}):'
    )

def parse_action(text):
    try:
        s = text.find('{')
        e = text.rfind('}') + 1
        p = _json.loads(text[s:e])
        action = p.get('intent', 'noop')
        conf = float(p.get('confidence', 0.5))
        return (action, conf) if action in VALID_ACTIONS else ('noop', conf)
    except Exception:
        return 'noop', 0.0

def calculate_advantages(rewards):
    r = torch.tensor(rewards, dtype=torch.float32)
    std = r.std()
    if std < 1e-8:
        return [0.0] * len(rewards)
    return ((r - r.mean()) / std).tolist()

print('Prompt builder and advantage calculator ready.')

In [ ]:
# Cell 6: GRPO Training Loop (3-Epoch Demo)
from torch.optim import AdamW
import matplotlib.pyplot as plt

optimizer = AdamW(model.parameters(), lr=5e-6)
GROUP_SIZE = 4
NUM_EPOCHS = 3
STEPS_PER_EPOCH = 5  # increase for real training

epoch_rewards = []

for epoch in range(NUM_EPOCHS):
    step_rewards = []
    for step in range(STEPS_PER_EPOCH):
        state = env.reset()
        prompt = build_prompt(state)
        inp = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=512).to(model.device)

        with torch.no_grad():
            outs = model.generate(
                **inp,
                max_new_tokens=64,
                temperature=0.8,
                do_sample=True,
                num_return_sequences=GROUP_SIZE,
                pad_token_id=tokenizer.pad_token_id,
            )

        plen = inp['input_ids'].shape[1]
        completions = [tokenizer.decode(o[plen:], skip_special_tokens=True) for o in outs]

        rewards = []
        for comp in completions:
            action, conf = parse_action(comp)
            _, reward = env.step(action)
            rewards.append(reward + conf * 0.5)

        advantages = calculate_advantages(rewards)

        optimizer.zero_grad()
        for comp, adv in zip(completions, advantages):
            if adv == 0.0:
                continue
            full_text = prompt + comp
            enc = tokenizer(full_text, return_tensors='pt', truncation=True, max_length=768).to(model.device)
            labels = enc['input_ids'].clone()
            labels[:, :plen] = -100
            loss = model(**enc, labels=labels).loss * adv
            loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        step_rewards.append(sum(rewards) / len(rewards))

    mean_r = sum(step_rewards) / len(step_rewards)
    epoch_rewards.append(mean_r)
    print('Epoch', epoch + 1, '/', NUM_EPOCHS, '| Mean Reward:', round(mean_r, 3))

# Plot results
plt.figure(figsize=(8, 4))
plt.plot(range(1, NUM_EPOCHS + 1), epoch_rewards, marker='o', color='lime', linewidth=2)
plt.title('GRPO Training: Mean Reward per Epoch')
plt.xlabel('Epoch')
plt.ylabel('Mean Reward')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('grpo_training_results.png', dpi=150)
plt.show()
print('Training complete!')